# Jupiter flux review — phase2 QA Zarr

Interactive summary of Stokes I flux toward **Jupiter** in pipeline QA Zarr stores
(`pipelineQA-phase2-I-NoTaper-Robust-0-*.zarr`).

For the selected observation day:

1. **Ephemeris** — Astropy `get_body("jupiter", …)` at the **first** dataset time step.
2. **Dynamic spectrum** — flux vs LST × frequency at that fixed RA/Dec via
   `ds.radport.dynamic_spectrum` (pixel tracking follows Earth rotation).
3. **Sky view** — click a cell in the dynamic spectrum to show that time/frequency
   slice in `astrowidget.SkyWidget`, centered on Jupiter at observation start.

Paths and Zarr naming follow `pipeline_qa_check_phase2.ipynb`.

Launch with: `pixi run jupyter lab`

In [1]:
from dataclasses import replace
from pathlib import Path

from ovro_lwa_portal.viz.pipeline_qa import PipelineQAConfig

# Edit before running if your staging paths differ.
ZARR_ROOT = Path("/fast/claw")
I_QA_ZARR_STEM = "pipelineQA-phase2-I-NoTaper-Robust-0"

QA_CONFIG = replace(
    PipelineQAConfig.phase2_default(),
    zarr_root=ZARR_ROOT,
    i_qa_zarr_stem=I_QA_ZARR_STEM,
)

In [2]:
import warnings

warnings.filterwarnings("ignore")

import math
from collections.abc import Callable

import ipywidgets as widgets
import numpy as np
import ovro_lwa_portal as ovro
import panel as pn
import param
import xarray as xr
import astropy.units as u
from astropy.coordinates import SkyCoord, get_body
from astropy.time import Time
from astropy.utils.iers import conf as iers_conf
from astrowidget import SkyWidget
from bokeh.events import Tap
from bokeh.models import ColumnDataSource, FixedTicker, HoverTool, LinearColorMapper
from bokeh.palettes import Inferno256
from bokeh.plotting import figure

from ovro_lwa_portal.viz.pipeline_qa import PipelineQAConfig
from ovro_lwa_portal.viz.pipeline_qa_app import (
    ACTIVITY_LOG_HEIGHT_PX,
    ScrollLog,
    _format_activity_log_html,
    _patch_astrowidget_get_wcs,
    _push_panel_layout,
    _schedule_ipython_main,
)

JUPITER_SKY_FOV_DEG = 10.0

_patch_astrowidget_get_wcs()
pn.extension("bokeh", sizing_mode="stretch_width")

In [3]:
def list_phase2_i_qa_zarrs(config: PipelineQAConfig) -> list[Path]:
    """Return sorted Stokes I phase2 QA Zarr paths under ``config.zarr_root``."""
    pattern = f"{config.i_qa_zarr_stem}-*.zarr"
    return sorted(config.zarr_root.glob(pattern))


def zarr_path_to_day(path: Path, *, stem: str) -> str:
    """Parse ``YYYY-MM-DD`` from a QA Zarr directory name."""
    day_tag = path.name.removeprefix(f"{stem}-").removesuffix(".zarr")
    if len(day_tag) != 8 or not day_tag.isdigit():
        return path.name
    return f"{day_tag[:4]}-{day_tag[4:6]}-{day_tag[6:8]}"


def jupiter_at_observation_start(ds: xr.Dataset) -> SkyCoord:
    """Jupiter FK5 coordinates at the first time sample in the dataset."""
    mjd = float(np.asarray(ds.coords["time"].values, dtype=np.float64)[0])
    orig = iers_conf.auto_download
    try:
        iers_conf.auto_download = False
        t0 = Time(mjd, format="mjd", scale="utc")
        return get_body("jupiter", t0)
    finally:
        iers_conf.auto_download = orig


def lst_hours_for_dataset(ds: xr.Dataset) -> np.ndarray:
    """Mean local sidereal time (hours) for each dataset time sample."""
    from astropy.coordinates import EarthLocation

    observatory = EarthLocation.of_site("ovro")
    mjd = np.asarray(ds.coords["time"].values, dtype=np.float64)
    orig = iers_conf.auto_download
    try:
        iers_conf.auto_download = False
        times = Time(mjd, format="mjd", scale="utc")
        lst_deg = np.asarray(times.sidereal_time("mean", longitude=observatory.lon).deg)
    finally:
        iers_conf.auto_download = orig
    return np.mod(lst_deg / 15.0, 24.0)


def _heatmap_index_from_coord(coord: float, n: int) -> int:
    if n <= 0:
        return 0
    return int(np.clip(int(np.floor(float(coord))), 0, n - 1))


def _format_lst_hour_label(lst_hour: float) -> str:
    hour = int(round(float(lst_hour))) % 24
    return f"{hour:02d}h"


def _color_mapper(values: np.ndarray) -> LinearColorMapper:
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        return LinearColorMapper(palette=Inferno256, low=0.0, high=1.0)
    lo, hi = np.percentile(finite, [2, 98])
    if hi <= lo:
        hi = lo + 1.0
    return LinearColorMapper(
        palette=Inferno256,
        low=float(lo),
        high=float(hi),
        nan_color=(128, 128, 128, 0.4),
    )

In [4]:
class JupiterFluxReview(param.Parameterized):
    """Dynamic spectrum toward Jupiter with linked SkyWidget."""

    select_zarr = param.Selector(default=None, objects=[], doc="Phase2 QA Zarr store.")
    loading = param.Boolean(default=False)
    status = param.String(default="Select a QA Zarr store.")
    log_text = param.String(default="")

    def __init__(self, config: PipelineQAConfig, **params) -> None:
        self._config = config
        self._scroll_log = ScrollLog()
        self._zarr_paths: dict[str, Path] = {}
        self._dataset: xr.Dataset | None = None
        self._dynspec: xr.DataArray | None = None
        self._jupiter: SkyCoord | None = None
        self._lst_hours: np.ndarray | None = None
        self._freq_mhz: np.ndarray | None = None
        self._sky_widget: SkyWidget | None = None
        self._time_idx = 0
        self._freq_idx = 0

        zarr_paths = list_phase2_i_qa_zarrs(config)
        if not zarr_paths:
            labels: list[str] = []
        else:
            labels = []
            for path in zarr_paths:
                day = zarr_path_to_day(path, stem=config.i_qa_zarr_stem)
                label = f"{day} ({path.name})"
                labels.append(label)
                self._zarr_paths[label] = path

        default = labels[-1] if labels else None
        super().__init__(select_zarr=default, **params)
        self.param.select_zarr.objects = labels

        self._heatmap_pane = pn.pane.Bokeh(height=420, sizing_mode="stretch_width")
        self._sky_container = widgets.VBox(
            children=[widgets.HTML("<i>Sky view loads after selecting a Zarr store.</i>")],
            layout=widgets.Layout(width="100%", min_height="620px"),
        )
        self._sky_pane = pn.pane.IPyWidget(self._sky_container, height=620, sizing_mode="stretch_width")
        self._status_pane = pn.pane.Markdown("")
        self._log_pane = pn.pane.HTML(
            _format_activity_log_html(""),
            sizing_mode="stretch_width",
            height=ACTIVITY_LOG_HEIGHT_PX,
        )
        self._selector = pn.widgets.Select.from_param(
            self.param.select_zarr,
            name="Phase2 QA Zarr (Stokes I)",
            width=520,
        )
        self._spinner = pn.indicators.LoadingSpinner(value=False, size=24, name="")
        self._layout = pn.Column(
            pn.Row(self._selector, self._spinner, margin=(0, 0, 8, 0)),
            pn.Column(
                pn.pane.Markdown("**Activity log**"),
                self._log_pane,
                sizing_mode="stretch_width",
            ),
            self._status_pane,
            self._heatmap_pane,
            self._sky_pane,
            sizing_mode="stretch_width",
            max_width=1048,
        )
        self.param.watch(self._on_select_zarr, "select_zarr")
        if labels:
            self._log(f"Found {len(labels)} phase2 QA Zarr store(s).")
        elif not labels:
            self._log("No phase2 QA Zarr stores found under the configured root.")
        if default is not None:
            self._load_selected_zarr()

    @property
    def panel(self) -> pn.Column:
        return self._layout

    def _on_select_zarr(self, *_events) -> None:
        self._load_selected_zarr()

    def _set_status(self, text: str) -> None:
        self.status = text
        self._status_pane.object = text

    @param.depends("log_text", watch=True)
    def _sync_log_pane(self) -> None:
        self._log_pane.object = _format_activity_log_html(self.log_text)

    def _sync_log(self) -> None:
        self.log_text = self._scroll_log.text

    def _log(self, message: str) -> None:
        self._scroll_log.append(message)
        self._sync_log()
        _push_panel_layout(self._layout, self._log_pane)

    def _load_selected_zarr(self) -> None:
        label = self.select_zarr
        if label is None:
            self._set_status("No phase2 QA Zarr stores found under the configured root.")
            return
        path = self._zarr_paths[label]
        self.loading = True
        self._spinner.value = True
        self._log(f"Loading {path.name}…")
        self._set_status("Loading…")
        _push_panel_layout(self._layout, self._status_pane, self._spinner, self._log_pane)

        def _work() -> None:
            try:
                _schedule_ipython_main(
                    lambda: self._log(f"Opening Stokes I Zarr at {path}…")
                )
                ds = ovro.open_dataset(path, chunks="auto").chunk({"l": 512, "m": 512})
                _schedule_ipython_main(
                    lambda: self._log(
                        f"Opened Zarr ({int(ds.sizes['time'])} times, "
                        f"{int(ds.sizes['frequency'])} frequencies, "
                        f"{int(ds.sizes['l'])}×{int(ds.sizes['m'])} pixels)."
                    )
                )
                _schedule_ipython_main(
                    lambda: self._log("Computing Jupiter ephemeris at observation start…")
                )
                jupiter = jupiter_at_observation_start(ds)
                _schedule_ipython_main(
                    lambda: self._log(
                        "Extracting dynamic spectrum toward Jupiter "
                        f"(RA={float(jupiter.ra.deg):.3f}°, Dec={float(jupiter.dec.deg):.3f}°)…"
                    )
                )
                dynspec = ds.radport.dynamic_spectrum(
                    ra=float(jupiter.ra.deg),
                    dec=float(jupiter.dec.deg),
                )
                lst_hours = lst_hours_for_dataset(ds)
                freq_mhz = np.asarray(ds.coords["frequency"].values, dtype=np.float64) / 1e6
            except Exception as exc:
                _schedule_ipython_main(
                    lambda: self._finish_load(None, None, None, None, None, exc)
                )
                return
            _schedule_ipython_main(
                lambda: self._finish_load(ds, dynspec, jupiter, lst_hours, freq_mhz, None)
            )

        import threading

        threading.Thread(target=_work, daemon=True).start()

    def _finish_load(
        self,
        ds: xr.Dataset | None,
        dynspec: xr.DataArray | None,
        jupiter: SkyCoord | None,
        lst_hours: np.ndarray | None,
        freq_mhz: np.ndarray | None,
        error: BaseException | None,
    ) -> None:
        self.loading = False
        self._spinner.value = False
        if error is not None:
            self._log(f"ERROR: {error}")
            self._set_status(f"**Load failed:** {error}")
            _push_panel_layout(self._layout, self._status_pane, self._spinner, self._log_pane)
            return

        assert ds is not None and dynspec is not None
        assert jupiter is not None
        assert lst_hours is not None and freq_mhz is not None

        self._dataset = ds
        self._dynspec = dynspec
        self._jupiter = jupiter
        self._lst_hours = lst_hours
        self._freq_mhz = freq_mhz

        jupiter_ra = jupiter.ra.to_string(unit=u.hour, precision=1)
        jupiter_dec = jupiter.dec.to_string(unit=u.deg, precision=1)
        self._set_status(
            f"**Jupiter at observation start:** RA={jupiter_ra}, Dec={jupiter_dec} · "
            f"{int(ds.sizes['time'])} times × {int(ds.sizes['frequency'])} frequencies · "
            "Click the dynamic spectrum to center the sky view on Jupiter."
        )

        self._mount_sky_widget(ds)
        self._time_idx, self._freq_idx = self._default_slice(dynspec.values)
        self._heatmap_pane.object = self._build_dynspec_figure(dynspec.values)
        self._update_sky(self._time_idx, self._freq_idx)
        self._log(
            f"Ready — Jupiter dynamic spectrum and sky view loaded for "
            f"{self._zarr_paths[self.select_zarr].name}."
        )
        _push_panel_layout(
            self._layout, self._status_pane, self._heatmap_pane, self._sky_pane, self._log_pane
        )

    def _default_slice(self, values: np.ndarray) -> tuple[int, int]:
        finite = np.argwhere(np.isfinite(values))
        if finite.size:
            t_idx, f_idx = finite[len(finite) // 2]
            return int(t_idx), int(f_idx)
        return 0, 0

    def _mount_sky_widget(self, ds: xr.Dataset) -> None:
        widget = SkyWidget()
        widget.colormap = "inferno"
        widget.background_survey = ""
        widget.invert_horizontal_pan = True
        max_size = max(256, int(ds.sizes["l"]) // 2)
        widget.set_dataset(ds, max_size=max_size)
        self._sky_widget = widget
        self._sky_container.children = [widget]

    def _update_sky(self, time_idx: int, freq_idx: int) -> None:
        widget = self._sky_widget
        jupiter = self._jupiter
        if widget is None or jupiter is None:
            return
        widget.update_slice(
            time_idx=int(time_idx),
            freq_idx=int(freq_idx),
            center=jupiter,
            fov=JUPITER_SKY_FOV_DEG * u.deg,
            percentile_low=2,
            percentile_high=98,
        )
        send_state = getattr(widget, "send_state", None)
        if callable(send_state):
            send_state()

    def _on_heatmap_tap(self, time_idx: int, freq_idx: int) -> None:
        self._time_idx = time_idx
        self._freq_idx = freq_idx
        if self._jupiter is None:
            return
        ra = self._jupiter.ra.to_string(unit=u.hour, precision=1)
        dec = self._jupiter.dec.to_string(unit=u.deg, precision=1)
        lst = _format_lst_hour_label(float(self._lst_hours[time_idx]))
        freq = float(self._freq_mhz[freq_idx])
        self._set_status(
            f"**Selected slice:** LST {lst}, {freq:.1f} MHz · "
            f"Jupiter (t₀) RA={ra}, Dec={dec}"
        )
        self._log(
            f"Sky view updated — time {time_idx}, freq {freq_idx} "
            f"({freq:.1f} MHz), FOV {JUPITER_SKY_FOV_DEG:.0f}°."
        )
        self._update_sky(time_idx, freq_idx)
        _push_panel_layout(self._layout, self._status_pane, self._sky_pane, self._log_pane)

    def _build_dynspec_figure(self, values: np.ndarray):
        n_times, n_freqs = values.shape
        mapper = _color_mapper(values)
        image = np.clip(
            (np.nan_to_num(values, nan=mapper.low) - mapper.low) / (mapper.high - mapper.low),
            0,
            1,
        )
        image_u8 = (image * 255).astype(np.uint8)

        plot = figure(
            width=1000,
            height=400,
            title="Dynamic spectrum toward Jupiter (RA/Dec at observation start)",
            x_range=(0, n_times),
            y_range=(0, n_freqs),
            tools="pan,wheel_zoom,reset,tap",
            active_drag="pan",
            active_tap="tap",
        )
        plot.image(image=[image_u8], x=0, y=0, dw=n_times, dh=n_freqs, palette=Inferno256)

        time_idx, freq_idx = np.meshgrid(
            np.arange(n_times, dtype=int),
            np.arange(n_freqs, dtype=int),
            indexing="ij",
        )
        flat_time = time_idx.ravel()
        flat_freq = freq_idx.ravel()
        hover_src = ColumnDataSource(
            data={
                "x": flat_time + 0.5,
                "y": flat_freq + 0.5,
                "time_idx": flat_time,
                "freq_idx": flat_freq,
                "lst_hour": [
                    _format_lst_hour_label(float(h))
                    for h in self._lst_hours[flat_time]
                ],
                "freq_mhz": self._freq_mhz[flat_freq],
                "flux_jy": values.ravel(),
            }
        )
        hover_renderer = plot.rect(
            x="x",
            y="y",
            width=1,
            height=1,
            source=hover_src,
            fill_alpha=0,
            line_alpha=0,
        )
        plot.add_tools(
            HoverTool(
                renderers=[hover_renderer],
                tooltips=[
                    ("LST hour", "@lst_hour"),
                    ("Freq (MHz)", "@freq_mhz{0.1}"),
                    ("Time idx", "@time_idx"),
                    ("Freq idx", "@freq_idx"),
                    ("Flux (Jy/beam)", "@flux_jy{0.3g}"),
                ],
            )
        )

        def _axis_ticks(n: int, values: np.ndarray, fmt: Callable) -> tuple[list[float], dict[float, str]]:
            step = 1 if n <= 24 else int(np.ceil(n / 24))
            indices = range(0, n, step)
            ticks = [i + 0.5 for i in indices]
            labels = {tick: fmt(values[i]) for tick, i in zip(ticks, indices, strict=True)}
            return ticks, labels

        x_ticks, x_labels = _axis_ticks(n_times, self._lst_hours, _format_lst_hour_label)
        y_ticks, y_labels = _axis_ticks(n_freqs, self._freq_mhz, lambda v: f"{float(v):.1f}")
        plot.xaxis.ticker = FixedTicker(ticks=x_ticks)
        plot.yaxis.ticker = FixedTicker(ticks=y_ticks)
        plot.xaxis.major_label_overrides = x_labels
        plot.yaxis.major_label_overrides = y_labels
        plot.xaxis.axis_label = "LST hour"
        plot.yaxis.axis_label = "Frequency (MHz)"
        plot.xaxis.major_label_orientation = math.pi / 4

        def _on_tap(event: Tap) -> None:
            if event.x is None or event.y is None:
                return
            t_idx = _heatmap_index_from_coord(event.x, n_times)
            f_idx = _heatmap_index_from_coord(event.y, n_freqs)
            _schedule_ipython_main(lambda: self._on_heatmap_tap(t_idx, f_idx))

        plot.on_event(Tap, _on_tap)
        return plot

In [5]:
review = JupiterFluxReview(QA_CONFIG)
review.panel

Column(max_width=1048, sizing_mode='stretch_width')
    [0] Row(margin=(0, 0, 8, 0), sizing_mode='stretch_width')
        [0] Select(description='Phase2 QA Zarr store.', name='Phase2 QA Zarr (..., options=OrderedDict({'2024-12-18 (...]), value='2024-12-28 (pipelineQA-ph..., width=520)
        [1] LoadingSpinner(size=24, value=True)
    [1] Column(sizing_mode='stretch_width')
        [0] Markdown(str, sizing_mode='stretch_width')
        [1] HTML(str, height=150, sizing_mode='stretch_width')
    [2] Markdown(str, sizing_mode='stretch_width')
    [3] Bokeh(None, height=420, sizing_mode='stretch_width')
    [4] IPyWidget(VBox, height=620, sizing_mode='stretch_width')

Extracting tracked pixels...
